In [1]:
import pandas as pd
import torch
import torch.nn as nn
import math
import numpy as np
import random
import time
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import IterableDataset, DataLoader
from transformers import MarianTokenizer

In [2]:
import os
os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'


tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

print("词表大小:", len(tokenizer)) # 这里应该输出 65000 左右
print("PAD_IDX:", tokenizer.pad_token_id)

词表大小: 65001
PAD_IDX: 65000


C:\Users\yixingll\miniconda3\envs\d2l\lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
import os

# 1. 配置路径 (替换成你实际的文件路径)
EN_INPUT = r'data\train.en'
ZH_INPUT = r'data\train.zh'

EN_OUTPUT = r'data\train_mini.en'
ZH_OUTPUT = r'data\train_mini.zh'

# 2. 设置目标数量和长度限制
TARGET_LINES = 500000  # 我们只要 50 万行
MAX_CHARS = 150        # 如果一行英文字符数超过 150，或者中文字符数超过 150，直接扔掉！
                       # (150 个字符通常对应 30-50 个 Token，远低于你的 max_length=128，非常安全)

print("开始过滤与切分数据集...")

kept_lines = 0
total_read = 0

# 采用流式读取，不吃内存
with open(EN_INPUT, 'r', encoding='utf-8') as f_en_in, \
     open(ZH_INPUT, 'r', encoding='utf-8') as f_zh_in, \
     open(EN_OUTPUT, 'w', encoding='utf-8') as f_en_out, \
     open(ZH_OUTPUT, 'w', encoding='utf-8') as f_zh_out:
    
    for en_line, zh_line in zip(f_en_in, f_zh_in):
        total_read += 1
        
        # 去掉首尾空格
        en_line = en_line.strip()
        zh_line = zh_line.strip()
        
        # 过滤条件：不能为空，且长度不能太长
        if en_line and zh_line and len(en_line) < MAX_CHARS and len(zh_line) < MAX_CHARS:
            f_en_out.write(en_line + '\n')
            f_zh_out.write(zh_line + '\n')
            kept_lines += 1
            
        # 如果达到了我们要的数量，就提前下班
        if kept_lines >= TARGET_LINES:
            break
            
        # 播报一下进度
        if total_read % 100000 == 0:
            print(f"已读取 {total_read} 行，已保存 {kept_lines} 行...")

print(f"搞定！成功从 {total_read} 行原始数据中，提取了 {kept_lines} 行优质短句！")
print(f"你的迷你数据集已保存在: {EN_OUTPUT} 和 {ZH_OUTPUT}")

In [3]:
class FastStreamDataset(IterableDataset):
    def __init__(self, en_filepath, zh_filepath):
        super().__init__()
        self.en_filepath = en_filepath
        self.zh_filepath = zh_filepath

    def __iter__(self):
        with open(self.en_filepath, 'r', encoding='utf-8') as f_en, \
             open(self.zh_filepath, 'r', encoding='utf-8') as f_zh:
            for en_line, zh_line in zip(f_en, f_zh):
                if not en_line.strip() or not zh_line.strip():
                    continue
                # 只把纯文本吐出去，不消耗 CPU 算力
                yield en_line.strip(), zh_line.strip()

def fast_collate_fn(batch):
    en_texts, zh_texts = zip(*batch)

    # 处理英文
    tokenizer.src_lang = "eng_Latn"
    en_encodings = tokenizer(
        list(en_texts), padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    src_padded = en_encodings['input_ids']

    # 处理中文
    tokenizer.src_lang = "zho_Hans"
    zh_encodings = tokenizer(
        list(zh_texts), padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    tgt_padded = zh_encodings['input_ids']

    # 🚨 【核心修复】：为中文目标序列强行插入起始符 (BOS)
    bsz = tgt_padded.size(0)
    # MarianTokenizer 的 pad_token_id 65000 经常被兼用来做 Decoder_start_token
    bos_tokens = torch.full((bsz, 1), tokenizer.pad_token_id, dtype=torch.long)
    # 拼装：[BOS, 词1, 词2, EOS, PAD...]
    tgt_padded = torch.cat([bos_tokens, tgt_padded], dim=1)

    return src_padded, tgt_padded


class MiniTranslationDataset(Dataset):
    def __init__(self, filepath):
        """
        一次性将整个极小数据集读入内存
        """
        self.data = []
        
        print(f"正在加载数据集: {filepath} ...")
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # cmn.txt 的标准格式是：English + \t + Chinese + \t + Attribution
                parts = line.split('\t')
                if len(parts) >= 2:
                    en_text = parts[0].strip()
                    zh_text = parts[1].strip()
                    self.data.append((en_text, zh_text))
                    
        print(f"✅ 加载成功！共读取了 {len(self.data)} 对中英句子。")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 每次返回一个元组 (英文文本, 中文文本)
        return self.data[idx]

# ==========================================
# 组装 DataLoader
# ==========================================

# 1. 实例化 Dataset
# 注意替换成你实际的文件路径
mini_dataset = MiniTranslationDataset(r'data\cmn.txt')

# 2. 划分训练集和验证集 (比如 90% 训练，10% 验证)
train_size = int(0.9 * len(mini_dataset))
val_size = len(mini_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(mini_dataset, [train_size, val_size])

print(f"切分完毕：训练集 {len(train_dataset)} 条，验证集 {len(val_dataset)} 条。")

# 3. 实例化 DataLoader
# 💡 这里直接复用你之前写好的 fast_collate_fn 来做 Tokenize 和 Padding
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True,          # 内存数据集可以开启 shuffle 打乱顺序，有利于模型收敛
    collate_fn=fast_collate_fn, 
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_size=32, 
    shuffle=False, 
    collate_fn=fast_collate_fn, 
    pin_memory=True
)

正在加载数据集: data\cmn.txt ...
✅ 加载成功！共读取了 29909 对中英句子。
切分完毕：训练集 26918 条，验证集 2991 条。


import torch
from torch.utils.data import IterableDataset, DataLoader

# ==========================================
# 1. 极速版流式数据集 (只读字符串，不在这里切词)
# ==========================================
class FastStreamDataset(IterableDataset):
    def __init__(self, en_filepath, zh_filepath):
        super().__init__()
        self.en_filepath = en_filepath
        self.zh_filepath = zh_filepath

    def __iter__(self):
        with open(self.en_filepath, 'r', encoding='utf-8') as f_en, \
             open(self.zh_filepath, 'r', encoding='utf-8') as f_zh:
            for en_line, zh_line in zip(f_en, f_zh):
                if not en_line.strip() or not zh_line.strip():
                    continue
                # 只把纯文本吐出去，不消耗 CPU 算力
                yield en_line.strip(), zh_line.strip()

# ==========================================
# 2. 批量分词函数 (利用 Rust 底层引擎瞬间处理 16 句话)
# ==========================================
def fast_collate_fn(batch):
    en_texts, zh_texts = zip(*batch)

    # 处理英文
    tokenizer.src_lang = "eng_Latn"
    en_encodings = tokenizer(
        list(en_texts), padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    src_padded = en_encodings['input_ids']

    # 处理中文
    tokenizer.src_lang = "zho_Hans"
    zh_encodings = tokenizer(
        list(zh_texts), padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    tgt_padded = zh_encodings['input_ids']

    # 🚨 【核心修复】：为中文目标序列强行插入起始符 (BOS)
    bsz = tgt_padded.size(0)
    # MarianTokenizer 的 pad_token_id 65000 经常被兼用来做 Decoder_start_token
    bos_tokens = torch.full((bsz, 1), tokenizer.pad_token_id, dtype=torch.long)
    # 拼装：[BOS, 词1, 词2, EOS, PAD...]
    tgt_padded = torch.cat([bos_tokens, tgt_padded], dim=1)

    return src_padded, tgt_padded

# ==========================================
# 3. 重新组装 DataLoader
# ==========================================
print("正在组装训练集流水线...")
train_dataset = FastStreamDataset(
    en_filepath=r'data\train_mini.en', 
    zh_filepath=r'data\train_mini.zh'
)
# 注意把 collate_fn 换成了我们新写的 fast_collate_fn
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=32, 
    collate_fn=fast_collate_fn,
    num_workers=0,          # 雇佣 4 个 CPU 核心同时切词
    # prefetch_factor=2,      # 提前准备好接下来的 2 个 Batch
    pin_memory=True         # 加速数据从 CPU 内存转移到 GPU 显存的速度
)

print("正在组装验证集流水线...")
val_dataset = FastStreamDataset(
    en_filepath=r'data\val.en', 
    zh_filepath=r'data\val.zh'
)
val_dataloader = DataLoader(
    val_dataset, 
    batch_size=32, 
    collate_fn=fast_collate_fn,
    num_workers=0, 
    pin_memory=True
)

print("流水线装填完毕！")

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_seq_len, d_model)
        
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() 
                             * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]

        return self.dropout(x)
        

In [5]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout_layer=None):
    # 假设 Q, K, V 的形状都是 (batch_size, seq_len, d_k) 

    d_k = K.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e4)
    attn_weights = torch.softmax(scores, dim=-1)

    if dropout_layer is not None:
        attn_weights = dropout_layer(attn_weights)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.attn_dropout = nn.Dropout(dropout)
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask,
                                                                dropout_layer=self.attn_dropout)

        concat_output = attn_output.transpose(1, 2).contiguous().view(
                        batch_size, -1, self.d_model)

        output = self.W_o(concat_output)

        return output, attn_weights

In [7]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc2(self.dropout(self.relu(self.fc1(x))))
        return x

In [8]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # 【Pre-LN 修改】：先归一化，再注意力，再残差
        x_norm = self.norm1(x)
        attn_output, _ = self.self_attn(x_norm, x_norm, x_norm, mask=mask)
        x = x + self.dropout1(attn_output)
        
        # 【Pre-LN 修改】：先归一化，再FFN，再残差
        x_norm2 = self.norm2(x)
        ffn_output = self.ffn(x_norm2)
        x = x + self.dropout2(ffn_output)
        return x

In [9]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

    def forward(self, X, mask):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.d_model))
        for layer in self.layers:
            X = layer(X, mask)

        return self.norm(X)

In [10]:
def create_look_ahead_mask(size):
    mask = torch.ones(size, size)
    mask = torch.tril(mask)
    return mask

In [11]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout=dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # 1. 掩码自注意力 (Pre-LN)
        x_norm1 = self.norm1(x)
        attn1, _ = self.self_attn(x_norm1, x_norm1, x_norm1, mask=tgt_mask)
        x = x + self.dropout1(attn1)
        
        # 2. 交叉注意力 (Pre-LN)
        x_norm2 = self.norm2(x)
        attn2, _ = self.cross_attn(x_norm2, enc_output, enc_output, mask=src_mask)
        x = x + self.dropout2(attn2)
        
        # 3. FFN (Pre-LN)
        x_norm3 = self.norm3(x)
        ffn_output = self.ffn(x_norm3)
        x = x + self.dropout3(ffn_output)
        
        return x

In [12]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

    def forward(self, X, enc_output, src_mask, tgt_mask):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.d_model))
        for layer in self.layers:
            X = layer(X, enc_output, src_mask, tgt_mask)

        return self.norm(X)

In [13]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads = 8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()

        self.encoder = Encoder(src_vocab_size, d_model, num_heads,
                               d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads,
                               d_ff, num_layers, dropout)
        self.projection = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)

        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)

        logits = self.projection(dec_output)

        return logits

In [14]:
def make_src_mask(src):
    # 源语言没有序列掩码，只有 PAD 掩码
    src_mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)
    return src_mask

def make_tgt_mask(tgt):
    tgt_len = tgt.shape[1]
    # 🚨 终极修复：只需要下三角矩阵（防止看未来），不需要再屏蔽 PAD
    tgt_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
    return tgt_mask

In [15]:
class ScheduledOptim:
    def __init__(self, optimizer, d_model, warmup_steps):
        self._optimizer = optimizer
        self.d_model = d_model
        self.n_warmup_steps = warmup_steps
        self.n_current_steps = 0

    def zero_grad(self):
        self._optimizer.zero_grad()
        
    def step_and_update_lr(self):
        self._update_learning_rate()
        self._optimizer.step()

    def _get_lr_scale(self):
        
        scale = min(self.n_current_steps ** -0.5,
                    self.n_current_steps * self.n_warmup_steps ** -1.5)

        return (self.d_model ** -0.5) * scale
    def _update_learning_rate(self):

        self.n_current_steps += 1
        lr = self._get_lr_scale()

        for param_group in self._optimizer.param_groups:
            param_group['lr'] = lr

In [16]:
# 初始化一个 GradScaler 用来缩放梯度（防止 FP16 精度下梯度下溢）
scaler = torch.amp.GradScaler('cuda')

def train_step(model, src_batch, tgt_batch, optimizer, criterion):
    tgt_input = tgt_batch[:, :-1]
    tgt_label = tgt_batch[:, 1:]
    
    src_mask = make_src_mask(src_batch)
    tgt_mask = make_tgt_mask(tgt_input)

    optimizer.zero_grad()

    # 1. 前向传播（混合精度）
    with torch.amp.autocast('cuda'):
        logits = model(src_batch, tgt_input, src_mask, tgt_mask)
        logits_flat = logits.contiguous().view(-1, logits.size(-1))
        tgt_label_flat = tgt_label.contiguous().view(-1)
        loss = criterion(logits_flat, tgt_label_flat)
    
    # 2. 反向传播（放大梯度防下溢）
    scaler.scale(loss).backward()

    # 🚨 关键修复：梯度裁剪防爆！
    # 因为 scaler 把梯度放大了，我们必须先把它还原 (unscale) 才能正确裁剪
    scaler.unscale_(optimizer._optimizer)
    # 将所有超过 1.0 的梯度强行削平到 1.0
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # 3. 更新参数
    optimizer._update_learning_rate() 
    scaler.step(optimizer._optimizer) 
    scaler.update()                   

    return loss.item()

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
print(f"正在使用的计算设备: {device} ")

# 1. 超参数设置
SRC_VOCAB_SIZE = len(tokenizer)
TGT_VOCAB_SIZE = len(tokenizer)
D_MODEL = 256
NUM_HEADS = 4
NUM_LAYERS = 2
D_FF = 1024


EPOCHS = 10 
SAVE_STEPS = 5000 # 极其重要：每训练 5000 个批次，存档一次！

# 2. 实例化模型
model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE, 
    tgt_vocab_size=TGT_VOCAB_SIZE, 
    d_model=D_MODEL, 
    num_heads=NUM_HEADS, 
    num_layers=NUM_LAYERS, 
    d_ff=D_FF
).to(device)

# 3. 损失函数与优化器
PAD_IDX = tokenizer.pad_token_id
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

base_optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
optimizer = ScheduledOptim(base_optimizer, D_MODEL, warmup_steps=2000)

train_loss_history = []
val_loss_history = []

# 4. 训练模型
for epoch in range(EPOCHS):
    model.train() # 启用 Dropout

    start_time = time.time()
    total_loss = 0
    batch_count = 0 # 必须手动计数，不能用 len(dataloader)
    
    # 这里的 train_dataloader 是你在上一个 Cell 实例化的
    for batch_idx, (src_batch, tgt_batch) in enumerate(train_dataloader):
        
        # 将数据推到 GPU 显存
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        
        # 核心：前向传播 + 反向传播
        loss = train_step(model, src_batch, tgt_batch, optimizer, criterion)
        
        total_loss += loss
        batch_count += 1
        
        # --- [1] 进度播报 ---
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx} | 当前 Loss: {loss:.4f}")
            
        # --- [2] 防断电保命存档 + 验证集抽查 ---
        if batch_idx > 0 and batch_idx % SAVE_STEPS == 0:
            # A. 保存临时权重
            save_path = f"transformer_epoch_{epoch+1}_step_{batch_idx}.pt"
            torch.save(model.state_dict(), save_path)
            print(f" 已保存临时权重: {save_path}")
            
            # B. 抽查验证集，防止模型“死记硬背（过拟合）”
            model.eval() # 临时关闭 Dropout
            val_loss = 0
            val_batches = 0
            with torch.no_grad(): # 不计算梯度，节省显存
                for v_idx, (v_src, v_tgt) in enumerate(val_dataloader):
                    if v_idx >= 50: # 我们只抽查前 50 个验证批次，节省时间
                        break
                    v_src, v_tgt = v_src.to(device), v_tgt.to(device)
                    v_tgt_input = v_tgt[:, :-1]
                    v_tgt_label = v_tgt[:, 1:]
                    
                    # 生成掩码
                    v_src_mask = make_src_mask(v_src)
                    v_tgt_mask = make_tgt_mask(v_tgt_input)
                    
                    # 计算分数
                    logits = model(v_src, v_tgt_input, v_src_mask, v_tgt_mask)
                    loss_v = criterion(
                        logits.contiguous().view(-1, logits.size(-1)), 
                        v_tgt_label.contiguous().view(-1)
                    )
                    val_loss += loss_v.item()
                    val_batches += 1
                    
            print(f"第 {batch_idx} 步验证集平均 Loss: {val_loss/val_batches:.4f}")
            model.train() 

    end_time = time.time()
    epoch_duration = end_time - start_time
    
    # --- [3] 一个 Epoch 完整结束 ---
    avg_loss = total_loss / max(1, batch_count)
    print(f"Epoch {epoch+1} 彻底结束 | 平均 Loss: {avg_loss:.4f}")
    print(f"耗时: {epoch_duration:.2f} 秒 (约 {epoch_duration/60:.2f} 分钟)")
    
    torch.save(model.state_dict(), f"transformer_epoch_{epoch+1}_FINAL.pt")

正在使用的计算设备: cuda 
Epoch 1/10 | Batch 0 | 当前 Loss: 11.1039
Epoch 1/10 | Batch 50 | 当前 Loss: 10.9372
Epoch 1/10 | Batch 100 | 当前 Loss: 10.3953
Epoch 1/10 | Batch 150 | 当前 Loss: 8.9617
Epoch 1/10 | Batch 200 | 当前 Loss: 6.4173
Epoch 1/10 | Batch 250 | 当前 Loss: 3.7954
Epoch 1/10 | Batch 300 | 当前 Loss: 2.4235
Epoch 1/10 | Batch 350 | 当前 Loss: 1.8257
Epoch 1/10 | Batch 400 | 当前 Loss: 2.1035
Epoch 1/10 | Batch 450 | 当前 Loss: 1.6986
Epoch 1/10 | Batch 500 | 当前 Loss: 1.5506
Epoch 1/10 | Batch 550 | 当前 Loss: 1.4994
Epoch 1/10 | Batch 600 | 当前 Loss: 1.5365
Epoch 1/10 | Batch 650 | 当前 Loss: 1.5988
Epoch 1/10 | Batch 700 | 当前 Loss: 1.6844
Epoch 1/10 | Batch 750 | 当前 Loss: 1.5513
Epoch 1/10 | Batch 800 | 当前 Loss: 1.6424
Epoch 1 彻底结束 | 平均 Loss: 3.9283
耗时: 159.34 秒 (约 2.66 分钟)
Epoch 2/10 | Batch 0 | 当前 Loss: 1.5622
Epoch 2/10 | Batch 50 | 当前 Loss: 1.6549
Epoch 2/10 | Batch 100 | 当前 Loss: 1.7097
Epoch 2/10 | Batch 150 | 当前 Loss: 1.4875
Epoch 2/10 | Batch 200 | 当前 Loss: 1.8793
Epoch 2/10 | Batch 250 | 当前 

In [20]:
def translate_sentence(model, sentence, tokenizer, device, max_length=50):
    model.eval() 
    
    # 🚨 【核心修复 1】：强行把 Tokenizer 的状态切回英文！
    # 消除训练遗留的状态污染，让模型正确认识输入的英文。
    tokenizer.src_lang = "eng_Latn" 
    
    bos_token = tokenizer.pad_token_id  # 65000
    eos_token = tokenizer.eos_token_id  # 0
        
    src_tensor = tokenizer.encode(sentence, return_tensors="pt").to(device)
    # 起始符是 65000
    tgt_tensor = torch.tensor([[bos_token]], dtype=torch.long).to(device)
    
    with torch.no_grad():
        for i in range(max_length):
            src_mask = make_src_mask(src_tensor)
            tgt_mask = make_tgt_mask(tgt_tensor)
            
            output = model(src_tensor, tgt_tensor, src_mask, tgt_mask)
            next_token_logits = output[0, -1, :] 
            
            # 🚨 【核心修复 2】：强行禁止模型在翻译过程中输出 PAD (65000)
            # 把 PAD 的逻辑值设为负无穷，保证 argmax 绝对不会选到它
            next_token_logits[bos_token] = -float('inf')
            
            next_token = next_token_logits.argmax(dim=-1).item()
            
            next_tensor = torch.tensor([[next_token]], dtype=torch.long).to(device)
            tgt_tensor = torch.cat([tgt_tensor, next_tensor], dim=1)
            
            if next_token == eos_token:
                break
                
    # 跳过 bos_token，进行解码
    translated_ids = tgt_tensor[0, 1:].tolist() 
    translation = tokenizer.decode(translated_ids, skip_special_tokens=True)
    return translation

# 再次运行测试！
print("====== 🤖 模型翻译测试 ======")
for text in sentences_to_test:
    result = translate_sentence(model, text, tokenizer, device)
    print(f"🇬🇧 原文: {text}")
    print(f"🇨🇳 翻译: {result}")
    print("-" * 30)

====== 🤖 模型翻译测试 ======
🇬🇧 原文: Hello world.
🇨🇳 翻译: 
------------------------------
🇬🇧 原文: I love you.
🇨🇳 翻译: 
------------------------------
🇬🇧 原文: What is your name?
🇨🇳 翻译: ?
------------------------------
🇬🇧 原文: This is a good book.
🇨🇳 翻译: 
------------------------------
🇬🇧 原文: Where are you going?
🇨🇳 翻译: ?
------------------------------
